# Ensemble agent (notebook prototype)

This notebook walks through **data → Chroma retrieval → RAG-style prompting → frontier LLM → Modal specialist pricer** and benchmarks on the Hugging Face test split.

**Working directory:** run Jupyter with the **project root** or **`notebooks/`** as the current working directory. The next cell sets `root_dir = Path.cwd().resolve().parents[0]` when CWD is `notebooks/` (parent = repo root). If you launch from repo root, change that line to `root_dir = Path.cwd().resolve()` so `src/` and paths still resolve.

**Env / secrets**
- `HF_TOKEN` — Hugging Face (dataset + hub)
- `OPENAI_API_KEY` — only for the **Frontier (GPT-5.1)** cells
- Modal CLI auth — for **Specialist** (`pricer`, class `Pricer`)

**Time / cost**
- Building the Chroma index over the **full training set** is slow (many encoder passes); existing non-empty collection skips rebuild.
- `evaluate(..., size=250)` on the specialist = **250 Modal remote calls**.
- Benchmarking GPT on 250 rows incurs **API cost**; use the optional cell with `RUN_GPT_BENCHMARK = False` by default.

**Suggested run order:** Setup → Load dataset → Chroma + encoder → Build collection → Similarity + RAG helpers → Frontier GPT (optional) → Evaluation imports → Specialist → Benchmarks.


## Setup


In [1]:
from pathlib import Path
import sys

root_dir = Path.cwd().resolve().parents[0]
sys.path.insert(0, str(root_dir / "src"))
sys.path.insert(0, str(root_dir / "notebooks"))


In [2]:
import os

from dotenv import load_dotenv
from huggingface_hub import login
from tqdm.auto import tqdm

from deal_hunter.agents.items import Item


/Volumes/256 SSD/deal_hunter/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load dataset


In [3]:
load_dotenv(override=True)
hf_token = os.environ["HF_TOKEN"]
login(token=hf_token, add_to_git_credential=False)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
pricer_data = "Vishy08/pricer-data"
train, test_items, val = Item.from_hub(dataset_name=pricer_data)
print(
    f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test_items):,} test items"
)


Loaded 320,000 training items, 4,000 validation items, 8,000 test items


## ChromaDB


In [5]:
import chromadb

DB = str(root_dir / "notebooks" / "products_vectorstore")
chroma_client = chromadb.PersistentClient(path=DB)


### Encoder (`sentence-transformers/all-MiniLM-L6-v2`)


In [6]:
from sentence_transformers import SentenceTransformer

encoder_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7031.70it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
sentences = [item.test_prompt for item in test_items[:50]]
embeddings = encoder_model.encode(sentences=sentences)


### Optional: embedding check


In [8]:
type(embeddings)


numpy.ndarray

### Build Chroma collection


In [9]:
import re

def summarizer(item):
    question = "How much does this cost to the nearest dollar?\n\n"
    prefix = "\n\nPrice is $"
    summary = item.test_prompt.replace(question, "").replace(prefix, "")
    return f"{summary}"


def parse_price(text: object) -> float:
    s = str(text).replace("$", "").replace(",", "")
    m = re.search(r"[-+]?\d*\.?\d+", s)
    return float(m.group()) if m else 0.0


### Optional: summarizer output sample


In [10]:
summarizer(test_items[3000])


'POWERTEC 70107V 4 T-Fitting for Dust Collection Hose, 1 PK\nIntroducing the 4-Inch T-Fitting by POWERTEC. This handy t-shaped connector optimizes the performance of your dust collection system by allowing you to add branches (connections) to your setup without compromising its efficiency. This creates a secure, airtight connection to ensure that harmful dust, dirt and debris is sucked in and expelled directly into your dust collector – resulting in clean air and a clean work environment. This dust collector hose adapter is ideal for professionals, hobbyists and workshops where space constraints or machine design are a concern. T-Shaped Fitting Dimensions 4-Inch OD nominal ends measure O.D. and I.D. Premium Build The POWERTEC T-Fitting is made out of high quality ABS plastic. This lightweight material is commonly known for it’s super durability'

In [11]:
def build_product_collection(
    chroma_client, encoder_model, items, name="products", batch_size=1000
):
    collection = chroma_client.get_or_create_collection(name)
    try:
        if collection.count() > 0:
            return collection
    except Exception:
        existing = collection.get(limit=1, include=[])
        if existing.get("ids"):
            return collection

    with tqdm(
        total=len(items), desc=f"Chroma {name}:", unit="doc", dynamic_ncols=True
    ) as pbar:
        for start in range(0, len(items), batch_size):
            batch = items[start : start + batch_size]
            docs = [summarizer(item) for item in batch]
            embeddings = encoder_model.encode(docs).astype(float).tolist()
            metas = [{"category": item.category, "price": item.price} for item in batch]
            doc_ids = [f"{name}_{start + k}" for k in range(len(batch))]
            collection.upsert(
                documents=docs,
                metadatas=metas,
                ids=doc_ids,
                embeddings=embeddings,
            )
            pbar.update(len(batch))
    return collection


In [12]:
collection = build_product_collection(chroma_client, encoder_model, train[:])


## Similarity search


In [13]:
chroma_client.get_collection("products")


Collection(name=products)

In [14]:
# Long product blurb (realistic length) for a manual retrieval demo
text = [
    "POWERTEC 70107V 4 T-Fitting for Dust Collection Hose, 1 PK\nIntroducing the 4-Inch T-Fitting by POWERTEC. This handy t-shaped connector optimizes the performance of your dust collection system by allowing you to add branches (connections) to your setup without compromising its efficiency. This creates a secure, airtight connection to ensure that harmful dust, dirt and debris is sucked in and expelled directly into your dust collector – resulting in clean air and a clean work environment. This dust collector hose adapter is ideal for professionals, hobbyists and workshops where space constraints or machine design are a concern. T-Shaped Fitting Dimensions 4-Inch OD nominal ends measure O.D. and I.D. Premium Build The POWERTEC T-Fitting is made out of high quality ABS plastic. This lightweight material is commonly known for it’s super durability"
]
result = collection.query(query_texts=text, n_results=3)
print(result["documents"][0][:])


['TERABITHIA Mini 10inch Realistic Couple Girls Children Birthday Xmas Gift Reborn Baby Doll Silicone Vinyl Full Body Newborn Twins\nCute and realistic reborn doll full body are made of high quality silicone-like vinyl.Safe non-toxic,pure environmentally friendly materials. Can pose a lot like sitting and lying,can not speaking. Full reborn doll body can enter into the water. The doll will come with outfits just like the picture. It will be a growth partners of your baby and the one of your family. Wish you love and take care of her (him). *These dolls are not toys;they are fine collectibles to be enjoyed by adult collectors. Attention The default is the baby girl, if you need baby boy, please contact us via email directly. You will get baby doll *2; 2.Doll clothes(May be some differences between each batch);']


In [15]:
def find_similar(text):
    vector = encoder_model.encode(text)
    result = collection.query(
        query_embeddings=vector,
        include=["documents", "metadatas"],
        n_results=5,
    )
    document = result["documents"][0][:]
    price = [m["price"] for m in result["metadatas"][0][:]]
    return document, price


In [16]:
find_similar(text)


(['TERABITHIA Mini 10inch Realistic Couple Girls Children Birthday Xmas Gift Reborn Baby Doll Silicone Vinyl Full Body Newborn Twins\nCute and realistic reborn doll full body are made of high quality silicone-like vinyl.Safe non-toxic,pure environmentally friendly materials. Can pose a lot like sitting and lying,can not speaking. Full reborn doll body can enter into the water. The doll will come with outfits just like the picture. It will be a growth partners of your baby and the one of your family. Wish you love and take care of her (him). *These dolls are not toys;they are fine collectibles to be enjoyed by adult collectors. Attention The default is the baby girl, if you need baby boy, please contact us via email directly. You will get baby doll *2; 2.Doll clothes(May be some differences between each batch);'],
 [49.99])

## RAG-style context for LLM


In [17]:
def make_context(similars, prices):
    message = "Here are the similar products with prices, which can be helpful for context\n"
    for similar, price in zip(similars, prices):
        message += f"Related Product:\n{similar}\nPrice is ${price}\n\n"
    return message


In [18]:
similars, prices = find_similar(text)
context = make_context(similars, prices)
print(context)


Here are the similar products with prices, which can be helpful for context
Related Product:
TERABITHIA Mini 10inch Realistic Couple Girls Children Birthday Xmas Gift Reborn Baby Doll Silicone Vinyl Full Body Newborn Twins
Cute and realistic reborn doll full body are made of high quality silicone-like vinyl.Safe non-toxic,pure environmentally friendly materials. Can pose a lot like sitting and lying,can not speaking. Full reborn doll body can enter into the water. The doll will come with outfits just like the picture. It will be a growth partners of your baby and the one of your family. Wish you love and take care of her (him). *These dolls are not toys;they are fine collectibles to be enjoyed by adult collectors. Attention The default is the baby girl, if you need baby boy, please contact us via email directly. You will get baby doll *2; 2.Doll clothes(May be some differences between each batch);
Price is $49.99




In [19]:
def message_for(item, similars, prices):
    message = f"Estimate the price of the product. Respond with Price only No Explanation\n\n{summarizer(item)}"
    message += make_context(similars, prices)
    return [{"role": "user", "content": message}]


In [20]:
similars, prices = find_similar(summarizer(train[2345]))
print(message_for(train[2345], similars, prices))


[{'role': 'user', 'content': "Estimate the price of the product. Respond with Price only No Explanation\n\nMoen CSI Arris Modern Hand -Towel Ring, 6.50 x 3.15 x 5.71 inches, Brushed Nickel\nProduct Description Arris faucets and accessories feature a cylindrical look that's thoroughly modern. Sharp angles add distinctive contrast to the tubular lines that dominate each piece in this collection. Exposed pipes and industrial-chic finishes are just a few of the downtown loft-inspired design details that come to life in Arris faucets and accessories. Sharp angles and tubular lines dominate each piece in this modern collection. Amazon.com Modern style with Moen functionality ( view larger ). Exposed pipes and industrial-chic finishes are just a few of the downtown loft-inspired design details that come to life in Moen's Arris faucets and accessories. Sharp angles and tubular lines dominate each piece in this stylish and functional modern collection. For a coordinated look toHere are the simi

## GPT-5.1 + RAG


In [21]:
from litellm import completion

# Requires OPENAI_API_KEY and earlier load_dotenv() from dataset section
_ = os.environ["OPENAI_API_KEY"]


In [22]:
def gpt_5_1_rag(item):
    documents, prices = find_similar(summarizer(item))
    response = completion(
        model="gpt-5.1",
        messages=message_for(item, documents, prices),
        reasoning_effort="none",
        seed=42,
    )
    return response.choices[0].message.content


In [23]:
response = gpt_5_1_rag(train[56678])
print(response)


$189.99


## Evaluation utilities


In [24]:
import importlib

import pandas as pd

import deal_hunter.services.testing as testing

importlib.reload(testing)
# Do not import ``test`` from this module — it shadows the HF ``test`` split name.
from deal_hunter.services.testing import Tester, evaluate


### Optional: benchmark GPT on the test split (API cost)


In [25]:
RUN_GPT_BENCHMARK = False
N_EVAL_GPT = 250

if RUN_GPT_BENCHMARK:

    def gpt_5_1_rag_row(row):
        item = Item.model_validate(row.to_dict())
        return parse_price(gpt_5_1_rag(item))

    _test_df = pd.DataFrame([x.model_dump() for x in test_items]).reset_index(drop=True)
    _n = min(N_EVAL_GPT, len(_test_df))
    evaluate(gpt_5_1_rag_row, _test_df, title="GPT-5.1 RAG", size=_n)


## Specialist (Modal fine-tuned pricer)

Deployed from [`../src/deal_hunter/services/pricer.py`](../src/deal_hunter/services/pricer.py): Modal app name `pricer`, class `Pricer`.


In [26]:
import modal

Pricer = modal.Cls.from_name("pricer", "Pricer")
pricer = Pricer()


In [27]:
def specialist(item):
    return pricer.price.remote(summarizer(item))


In [28]:
specialist(train[56])


19.0

## Benchmarks (test split)


In [29]:
N_EVAL = 250
BENCHMARK_TITLE = "Llama_3.1_8B_Finetuned"


def llama8(row) -> float:
    item = Item.model_validate(row.to_dict())
    raw = specialist(item)
    return parse_price(raw)


test_df = pd.DataFrame([x.model_dump() for x in test_items]).reset_index(drop=True)
n = min(N_EVAL, len(test_df))

#uncomment for testing
# evaluate(llama8, test_df, title=BENCHMARK_TITLE, size=n)


## Ensemble Weights

In [30]:
import numpy as np 
import pandas as pd

from sklearn.linear_model import LinearRegression
from deal_hunter.agents.items import Item


In [31]:
def pred_specialist(item:Item) ->float:
    return float(specialist(item))   # uses Modal (pricer.py)

def pred_gpt(item:Item) -> float :
    return parse_price(gpt_5_1_rag(item))  #  uses liteLLM 


In [32]:
def build_meta_df(items,n=None):
    rows =[]
    seq = items if n is None else items[:n]
    for it in seq:
        rows.append({
            "price":float(it.price) ,
            "p_specialist":pred_specialist(it),
            "p_gpt":pred_gpt(it),
        })
    return pd.DataFrame(rows)

In [ ]:
build_meta_df(val,n=20)

In [34]:
#LR
meta_val = build_meta_df(val,n = 100)
X_val = meta_val[["p_specialist","p_gpt"]].values
y_val = meta_val["price"].values

reg = LinearRegression(fit_intercept = False, positive = True)
reg.fit(X_val,y_val)

w = reg.coef_.astype(float)
w = w/w.sum() if w.sum() > 0 else np.array([0.5,0.5])

w_specialist , w_gpt = w
print("learned Weights:",{"specialist":w_specialist,"gpt" : w_gpt})

learned Weights: {'specialist': np.float64(0.1884875810824194), 'gpt': np.float64(0.8115124189175806)}


In [35]:
def ensemble_model(item:Item)->float:
    ps = pred_specialist(item)
    pg = pred_gpt(item)
    return float(w_specialist*ps + w_gpt*pg)

In [ ]:
def ensemble_row(row) -> float:
    item = Item.model_validate(row.to_dict())
    return ensemble_model(item)

test_df = pd.DataFrame([x.model_dump() for x in test_items]).reset_index(drop=True)
evaluate(ensemble_row, test_df, title="Ensemble (LR weights)", size=250)